In [ ]:
#%pip install pyserial

In [ ]:
# %pip install scikit-learn

In [ ]:
# %pip install pandas
# %pip install torch
# %pip install pyserial


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


import serial
import threading
import time


### Loading the Model

In [ ]:
'''# Train REGULARIZED version
class EmotionCNN_Reg(nn.Module):
    def __init__(self, input_channels=14, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv1d(input_channels, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(2)

        self.fc1       = nn.Linear(32 * 31, 64)
        self.dropout1  = nn.Dropout(0.3)
        self.fc2       = nn.Linear(64, 32)
        self.dropout2  = nn.Dropout(0.5)
        self.fc3       = nn.Linear(32, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.dropout(x, 0.2)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

'''

In [ ]:
class EmotionCNN_RegV2(nn.Module):
    """
    Model architecture copying TinyVGG from: 
    https://poloclub.github.io/cnn-explainer/
    """
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.block_1 = nn.Sequential(
            nn.Conv1d(in_channels=input_shape, 
                      out_channels=hidden_units, 
                      kernel_size=3, 
                      stride=1, 
                      padding=1), 
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Conv1d(in_channels=hidden_units, 
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2,
                         stride=2) # default stride value is same as kernel_size
        )
        self.block_2 = nn.Sequential(
            nn.Conv1d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_units),
            nn.Conv1d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            # Where did this in_features shape come from? 
            # It's because each layer of our network compresses and changes the shape of our input data.
            nn.Linear(in_features=hidden_units*31 ,out_features=output_shape)
        )
    
    def forward(self, x: torch.Tensor):
        x = self.block_1(x)
        # print(x.shape)
        x = self.block_2(x)
        # print(x.shape)
        x = self.classifier(x)
        # print(x.shape)
        return x

In [ ]:
# Load the best CNN model

PATH= 'Ml-Models/emotionv2_tiny_vgg_updated_windowing_14features.pth'

model = EmotionCNN_RegV2(input_shape=14,
                            hidden_units=10,
                            output_shape=4)

model.load_state_dict(torch.load(PATH, weights_only=True))
model.eval()


### Live Data Streaming

In [ ]:
'''if torch.backends.mps.is_available():
    device = torch.device("mps")  

else:
    device = torch.device("cpu")''' 
device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
#global_mean = [0.000135, 0.000919, -0.001325, 0.000138, -8.5e-05, 0.001069, -0.001142, -0.001113, 0.003192, -0.000292, 0.000753, -0.000517, -0.003865, -0.002841]
#global_std  = [0.98748, 0.993947, 0.987439, 0.983629, 0.979366, 0.98413, 0.993598, 0.994275, 0.989484, 0.979971, 0.980799, 0.977189, 0.985282, 0.989895]

In [ ]:
SERIAL_COLS = ["t_ms", "ax1", "ay1", "az1", "roll1", "pitch1", "yaw1", "gx1", "gy1", "gz1",
               "ax2", "ay2", "az2", "roll2", "pitch2", "yaw2", "gx2", "gy2", "gz2", "emg1", "emg2"]

FEATURE_COLS = ["ax1", "ay1", "az1", "gx1", "gy1", "gz1",
                "ax2", "ay2", "az2", "gx2", "gy2", "gz2", "emg1", "emg2"]
                
FEATURE_INDICES = [SERIAL_COLS.index(c) for c in FEATURE_COLS]

In [ ]:
# Start Data streaming
# How is the live data coming in?

COM_PORT='/dev/cu.usbmodem21201'


window_size = 125
#overlap_size = int(0.25 * window_size)
step_size = 25 # or 32 depending on what model you are using.

live_buffer = []               # The shared "bowl"
buffer_lock = threading.Lock() # The "Pause" button to prevent data crashes

def data_collection_thread():
    # Open the serial port inside the producer
    ser = serial.Serial(COM_PORT, 115200)
    ser.reset_input_buffer() # Clear out old junk!
    #correct_columns=21 # for 20 feature model
    #correct_columns=15 # for 20 feature model

    correct_columns=len(FEATURE_COLS)
    

    print("Producer: Listening to Arduino at 30Hz...")
    
    while True:
        try:
            # new code to check if the line has the correct number of columns before next step
            raw_line = ser.readline().decode('utf-8').strip()
            #print(type(raw_line))
            sample_list=[]
            indi_params= raw_line.split(',')

            #dropped_indices = [6,7,8,15,16,17]
            #filtered_params = [indi_params[i] for i in range(len(indi_params)) if i not in dropped_indices]
            
            #print(len(indi_params)) ## Debugging

            ## For the code wthout clean column index slicing:
            #if len(filtered_params) == correct_columns:


            if len(indi_params) == len(SERIAL_COLS):
                ## USE THE NEXT LINE IF MODEL DOESNT USE ORIENTATION DATA.
                # This is for the script without the clean column index slicing;
                #sample_list = [float(x) for x in filtered_params] #changed this up to make sure its not a generator and its a list
                
                ## Code with clean slicing:
                sample_list=[float(indi_params[i]) for i in FEATURE_INDICES]

                
                # sample_list = [float(x) for x in indi_params]
                #print(sample_list)
                with buffer_lock:
                    live_buffer.append(sample_list)
            else:
                pass
                
        except Exception as e:
            pass


def inference_thread():
    class_labels = {0: "Distracted", 1: "Focus", 2: "Relaxed", 3: "Stress"}    
    print("Consumer: Waiting for 125 rows...")
    model.eval() 
    
    short_window_samples=window_size
    long_window_samples=window_size*12

    last_inference=time.time()


    while True:

        curr=time.time()
        #data_to_process = None
        
        # Pull from buffer
        with buffer_lock:
            buffer_len = len(live_buffer)
        
        if buffer_len < long_window_samples:
            print(f"Collecting data... {buffer_len}/{long_window_samples}")
            time.sleep(0.1)
            continue

        if curr-last_inference<5:
            time.sleep(0.05)
            continue
        last_inference= curr
    
        with buffer_lock:
            buffer_snapshot = list(live_buffer)

        # 1) SHORT WINDOW: Last 125 samples (5 sec)
        if len(buffer_snapshot) >= short_window_samples:
            short_data = buffer_snapshot[-short_window_samples:]
            short_pred, short_conf = process_window(short_data)
            print(f"[LAST 5s]  {class_labels.get(short_pred, 'Unknown')} | {short_conf:.2%}")

        # 2) LONG WINDOW: Last 1500 samples (60 sec)  
        if len(buffer_snapshot) >= long_window_samples:
            long_data = buffer_snapshot[-long_window_samples:]
            long_pred, long_conf = process_window(long_data)
            #print(f"[LAST 60s] {class_labels.get(long_pred, 'Unknown')} | {long_conf:.2%}")

def process_window(data_list):
    """Helper: Process a window of ANY size into model predictions"""
    #data_array = np.array(data_list, dtype=np.float32)[:, 1:]  # drop first column


    data_array = np.array(data_list, dtype=np.float32)  # Already 14 columns.
    
    # Normalize
    #mean = np.mean(data_array, axis=0)
    #std = np.std(data_array, axis=0) 
    #data_normalized = (data_array - mean) / (std + 1e-8)
    
    # Break into overlapping 125-sample windows
    windows = []
    step_size = 25  # 94
    #for i in range(0, len(data_normalized) - window_size + 1, step_size):
    for i in range(0, len(data_array) - window_size + 1, step_size):

        window = data_array[i:i + window_size]
        
        # Per-window normalization 
        w_mean = np.mean(window, axis=0, keepdims=True)
        w_std = np.std(window, axis=0, keepdims=True)
        window_norm = (window - w_mean) / (w_std + 1e-8)
        
        windows.append(window_norm)
        #windows.append(data_normalized[i:i+window_size])
    
    if not windows:
        return None, 0
    
    # Stack and predict
    windows = np.stack(windows)  # (N, 125, features)
    windows = np.transpose(windows, (0, 2, 1))  # (N, features, 125)
    input_tensor = torch.tensor(windows, dtype=torch.float32).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        avg_probs = probs.mean(dim=0)  # Average across all windows
        conf, pred_idx = torch.max(avg_probs, dim=0)
    
    return pred_idx.item(), conf.item()
'''        if data_to_process is not None:
            # Convert to numpy (Shape: 125, 21)
            print("IN HERE")
            data_array = np.array(data_to_process,dtype=np.float32)
            
            data_array = data_array[:, 1:] 

            

            mean = np.mean(data_array, axis=0)
            std = np.std(data_array, axis=0)
            data_normalized = (data_array - mean) / (std + 1e-8)

            input_tensor = torch.tensor(data_normalized, dtype=torch.float32)#Convert to Torch Tensor
            
            input_tensor = input_tensor.transpose(0, 1)
            
            input_tensor = input_tensor.unsqueeze(0).to(device)# making it (1, Channels, 125) 

            try:
                with torch.no_grad():
                    output = model(input_tensor)
                    
                    probabilities = torch.softmax(output, dim=1)
                    
                    conf, pred_idx = torch.max(probabilities, dim=1)
                    
                    prediction = pred_idx.item()
                    confidence = conf.item()

                print(f"Result: {class_labels.get(prediction, 'Unknown')} | Conf: {confidence:.2%}")
                
            except Exception as e:
                print(f"Inference Error: {e}")
                print(f"Check if input_tensor shape {input_tensor.shape} matches model!")

        else:
            print("Not enough data yet...")
            time.sleep(0.01)'''




# Create the worker threads
t1 = threading.Thread(target=data_collection_thread, daemon=True)
t2 = threading.Thread(target=inference_thread, daemon=True)

# Start them
t1.start()
t2.start()

# Keep the main script alive so the threads can run in the background
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Shutting down the system...")




[LAST 5s]  Distracted | 99.29%
[LAST 5s]  Stress | 85.62%
[LAST 5s]  Distracted | 97.89%
[LAST 5s]  Distracted | 99.98%
[LAST 5s]  Distracted | 100.00%
[LAST 5s]  Distracted | 92.73%
[LAST 5s]  Distracted | 100.00%
[LAST 5s]  Distracted | 99.22%
[LAST 5s]  Distracted | 100.00%
[LAST 5s]  Stress | 99.82%
[LAST 5s]  Stress | 98.25%
[LAST 5s]  Distracted | 99.95%
[LAST 5s]  Relaxed | 96.07%
[LAST 5s]  Stress | 97.89%
[LAST 5s]  Stress | 100.00%
[LAST 5s]  Distracted | 100.00%
[LAST 5s]  Distracted | 99.99%
[LAST 5s]  Distracted | 99.50%
[LAST 5s]  Focus | 100.00%
[LAST 5s]  Stress | 99.98%
[LAST 5s]  Relaxed | 86.82%
[LAST 5s]  Focus | 99.67%
[LAST 5s]  Stress | 99.98%
[LAST 5s]  Stress | 99.54%
[LAST 5s]  Distracted | 99.24%
[LAST 5s]  Focus | 99.97%
[LAST 5s]  Focus | 74.61%
[LAST 5s]  Focus | 99.94%
[LAST 5s]  Focus | 100.00%
[LAST 5s]  Relaxed | 99.93%
[LAST 5s]  Distracted | 100.00%
[LAST 5s]  Focus | 87.72%
[LAST 5s]  Stress | 79.53%
[LAST 5s]  Stress | 100.00%
[LAST 5s]  Focus | 98

### Data Processing

In [ ]:
# Call Data Processing Pipeline

### Model Prediction

In [ ]:
# Live Predictions

